# 10장. 웹 URL 기반 RAG와 웹 문서 요약

이 장은 PDF 교재의 `6.4 활용사례 - 실시간 정보 제공`과 `LLM/llm2.ipynb`의 `챗봇 예제(인터넷 URL정보 요약하기 + TAB추가)` 실습을 통합한 장입니다.

## 이 장에서 다루는 내용

1. 웹 URL 기반 RAG란 무엇인가
2. 실시간 정보 제공과 RAG
3. `WebBaseLoader`
4. BeautifulSoup/bs4의 역할
5. 웹 문서 로딩
6. 웹 문서 청크 분할
7. `OllamaEmbeddings`
8. FAISS 벡터스토어 생성
9. Retriever 구성
10. 검색 문서 포맷팅
11. URL 기반 RAG Chain
12. Gradio Blocks와 Tab UI
13. 웹 RAG의 한계와 주의사항

이 장의 목표는 사용자가 URL과 질문을 입력하면, 웹페이지 내용을 읽어 관련 정보를 검색하고 답변하는 RAG 앱을 만드는 것입니다.


## 10.1 웹 URL 기반 RAG란?

웹 URL 기반 RAG는 웹페이지를 외부 지식 소스로 사용하는 RAG입니다.

일반 RAG가 로컬 문서나 DB를 검색한다면, URL RAG는 사용자가 입력한 웹페이지 내용을 읽고 그 안에서 답을 찾습니다.

기본 흐름:

```text
URL 입력
  -> 웹페이지 로드
  -> 텍스트 추출
  -> 문서 청크 분할
  -> 임베딩
  -> FAISS 벡터스토어 생성
  -> 질문 기반 검색
  -> LLM 답변 생성
```

PDF 6.4의 `실시간 정보 제공` 예시와 연결됩니다. 예를 들어 최신 뉴스, 제품 소개 페이지, 공지사항 페이지를 읽고 사용자의 질문에 답할 수 있습니다.


## 10.2 웹 RAG가 유용한 경우

웹 URL 기반 RAG는 다음 상황에서 유용합니다.

- 특정 웹페이지 내용을 요약하고 싶을 때
- 긴 기사에서 질문에 해당하는 부분만 찾고 싶을 때
- 제품 소개 페이지를 기반으로 Q/A를 만들고 싶을 때
- 공지사항이나 문서 페이지를 빠르게 분석하고 싶을 때
- 로컬 파일로 저장하지 않고 URL만으로 실습하고 싶을 때

다만 웹페이지는 구조가 다양하고, 일부 페이지는 JavaScript 렌더링이나 로그인, 크롤링 차단이 있을 수 있습니다. 그래서 모든 웹사이트가 항상 잘 로드되는 것은 아닙니다.


## 10.3 설치 준비

URL RAG 예제에 필요한 주요 패키지는 다음과 같습니다.

| 패키지 | 역할 |
|---|---|
| `gradio` | 웹 UI 구성 |
| `beautifulsoup4` | HTML 파싱 보조 |
| `langchain-text-splitters` | 문서 청크 분할 |
| `langchain-community` | WebBaseLoader, FAISS, OllamaEmbeddings, Ollama |
| `faiss-cpu` | 벡터스토어 |
| `ollama` 또는 Ollama 설치 | 로컬 임베딩/LLM 실행 |

`WebBaseLoader`는 내부적으로 웹페이지를 가져오고 HTML을 처리합니다. `bs4`는 BeautifulSoup 기반 파싱 옵션을 줄 때 사용됩니다.


In [ ]:
# URL RAG 실습용 설치 예시입니다.
# 필요한 경우 주석을 해제하고 실행하세요.

# %pip install gradio beautifulsoup4
# %pip install langchain langchain-community langchain-core langchain-text-splitters
# %pip install faiss-cpu


## 10.4 모듈 import

`llm2.ipynb`의 URL RAG 예제는 다음 모듈을 사용합니다.

| 모듈 | 역할 |
|---|---|
| `gradio` | URL과 질문 입력 UI, 결과 출력 UI |
| `bs4` | HTML 파싱 옵션에 사용 |
| `RecursiveCharacterTextSplitter` | 웹 문서 청크 분할 |
| `WebBaseLoader` | 웹페이지 콘텐츠 로드 |
| `FAISS` | 벡터스토어 생성 |
| `OllamaEmbeddings` | Ollama 모델 기반 임베딩 |
| `Ollama` | 답변 생성 LLM |
| `ChatPromptTemplate` | 질문/context 프롬프트 구성 |
| `StrOutputParser` | LLM 출력 문자열 파싱 |


In [ ]:
# Gradio 웹 UI를 만들기 위해 사용합니다.
import gradio as gr

# BeautifulSoup HTML 파싱 관련 옵션에 사용합니다.
import bs4

# 텍스트를 재귀적으로 적절한 크기의 청크로 나눕니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 웹페이지 콘텐츠를 LangChain Document로 로드합니다.
from langchain_community.document_loaders import WebBaseLoader

# FAISS 벡터스토어입니다.
from langchain_community.vectorstores import FAISS

# Ollama 기반 임베딩 모델입니다.
from langchain_community.embeddings import OllamaEmbeddings

# Ollama LLM입니다.
from langchain_community.llms import Ollama

# 프롬프트 템플릿과 출력 파서입니다.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


## 10.5 URL 문서 로드와 Retriever 생성 함수

원본 노트북의 핵심 함수는 `load_and_retrieve_docs(url)`입니다.

이 함수가 하는 일:

1. URL을 받아 `WebBaseLoader`를 생성합니다.
2. 웹페이지 내용을 LangChain Document로 로드합니다.
3. 문서를 `RecursiveCharacterTextSplitter`로 청크 분할합니다.
4. `OllamaEmbeddings(model="gemma2")`로 임베딩 모델을 준비합니다.
5. `FAISS.from_documents()`로 벡터스토어를 만듭니다.
6. `as_retriever()`로 검색기를 반환합니다.

이 함수는 URL이 들어올 때마다 새 벡터스토어를 만듭니다. 실습에는 간단하지만, 같은 URL을 반복 사용할 경우 캐싱을 고려할 수 있습니다.


In [ ]:
# URL을 받아 웹 문서를 로드하고 Retriever를 반환하는 함수입니다.
def load_and_retrieve_docs(url):
    # 입력 URL을 WebBaseLoader에 전달합니다.
    loader = WebBaseLoader(
        web_paths=(url,),
        bs_kwargs=dict()
    )

    # 웹페이지 내용을 Document 객체로 로드합니다.
    docs = loader.load()

    # 문서를 1000자 크기, 200자 overlap으로 나눕니다.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    # Ollama 기반 임베딩 모델을 사용합니다.
    embeddings = OllamaEmbeddings(model="gemma2")

    # 분할된 문서를 FAISS 벡터스토어로 만듭니다.
    vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

    # Retriever로 변환해 반환합니다.
    return vectorstore.as_retriever()


## 10.6 웹 문서 포맷팅 함수

Retriever는 관련 Document 객체 목록을 반환합니다. LLM 프롬프트에 넣으려면 문서 내용을 하나의 문자열로 합쳐야 합니다.

원본 노트북의 `format_docs()` 함수는 각 문서의 `page_content`를 가져와 `\n\n`으로 연결합니다.

```python
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
```

이렇게 만든 문자열이 RAG 프롬프트의 `{context}`에 들어갑니다.


In [ ]:
# 검색된 문서의 본문을 하나의 context 문자열로 합칩니다.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


## 10.7 URL 기반 RAG Chain 함수

원본 노트북의 `rag_chain(url, question)` 함수는 URL과 질문을 받아 답변을 반환합니다.

함수 내부 흐름:

1. `load_and_retrieve_docs(url)`로 Retriever를 만듭니다.
2. `retriever.invoke(question)`으로 관련 문서를 검색합니다.
3. `format_docs(retrieved_docs)`로 context를 구성합니다.
4. `ChatPromptTemplate.from_template()`로 프롬프트를 만듭니다.
5. `Ollama(model='gemma3:4b')`로 LLM을 준비합니다.
6. `prompt | llm | StrOutputParser()`로 체인을 만듭니다.
7. 질문과 context를 넣어 답변을 생성합니다.

이 방식은 함수 하나 안에서 문서 로드부터 답변 생성까지 모두 처리합니다.


In [ ]:
# URL 기반 RAG 시스템 함수입니다.
def rag_chain(url, question):
    # URL 문서를 로드하고 Retriever를 생성합니다.
    retriever = load_and_retrieve_docs(url)

    # 사용자 질문과 관련 있는 문서를 검색합니다.
    retrieved_docs = retriever.invoke(question)

    # 검색된 문서들을 하나의 context 문자열로 포맷팅합니다.
    formatted_context = format_docs(retrieved_docs)

    # 질문과 context를 넣을 프롬프트 템플릿을 만듭니다.
    prompt = ChatPromptTemplate.from_template(
        "Question: {question}\n\nContext: {context}"
    )

    # 답변 생성용 Ollama 모델을 준비합니다.
    llm = Ollama(model='gemma3:4b')

    # 프롬프트, LLM, 출력 파서를 LCEL 방식으로 연결합니다.
    chain = prompt | llm | StrOutputParser()

    # 체인을 실행해 답변을 생성합니다.
    response = chain.invoke({
        "question": question,
        "context": formatted_context
    })

    # 최종 답변을 반환합니다.
    return response


## 10.8 프롬프트 개선 방향

원본 노트북의 프롬프트는 단순합니다.

```text
Question: {question}

Context: {context}
```

실제 RAG 품질을 높이려면 다음처럼 더 명확한 지시를 넣을 수 있습니다.

```text
당신은 웹 문서를 기반으로 질문에 답하는 AI 어시스턴트입니다.
아래 Context에 있는 정보만 사용하세요.
Context에 답이 없으면 모른다고 답하세요.

[Context]
{context}

[Question]
{question}
```

이런 지시는 모델이 웹페이지에 없는 내용을 추측하는 것을 줄이는 데 도움이 됩니다.


In [ ]:
# 개선된 URL RAG 함수 예시입니다.
def rag_chain_with_strict_prompt(url, question):
    # URL에서 Retriever를 생성합니다.
    retriever = load_and_retrieve_docs(url)

    # 질문 관련 문서를 검색합니다.
    retrieved_docs = retriever.invoke(question)
    formatted_context = format_docs(retrieved_docs)

    # 더 엄격한 RAG 프롬프트입니다.
    template = """
당신은 웹 문서를 기반으로 질문에 답하는 AI 어시스턴트입니다.
아래 Context에 있는 정보만 사용하세요.
Context에 답이 없으면 모른다고 답하세요.

[Context]
{context}

[Question]
{question}
"""

    # 프롬프트와 모델, 출력 파서를 연결합니다.
    prompt = ChatPromptTemplate.from_template(template)
    llm = Ollama(model='gemma3:4b')
    chain = prompt | llm | StrOutputParser()

    # 체인을 실행합니다.
    return chain.invoke({
        "question": question,
        "context": formatted_context
    })


## 10.9 Gradio Tabbed Interface

`llm2.ipynb`의 URL RAG 예제는 Gradio의 Tab UI를 사용합니다.

탭 구성:

| 탭 | 역할 |
|---|---|
| 질문과 답변 | URL과 질문을 입력해 RAG 답변을 받습니다. |
| 시각화(워드클라우드) | 향후 웹페이지 텍스트 시각화를 넣을 공간입니다. |

`gr.Blocks()` 안에서 `gr.Tab()`을 사용하면 하나의 앱 안에 여러 화면을 구성할 수 있습니다.


In [ ]:
# Gradio Tabbed Interface입니다.
with gr.Blocks() as iface:
    # 질문과 답변 탭입니다.
    with gr.Tab("질문과 답변"):
        gr.Interface(
            fn=rag_chain,
            inputs=["text", "text"],
            outputs="text",
            title="RAG Chain Question Answering",
            description="URL과 질문을 입력하면 RAG 체인에서 답변을 받을 수 있습니다."
        )

    # 시각화 탭입니다.
    with gr.Tab("시각화 (워드클라우드)"):
        gr.Markdown("이 탭은 시각화를 위한 공간입니다. 워드클라우드 기능이 여기에 추가될 예정입니다.")


In [ ]:
# Gradio 인터페이스를 실행합니다.
# debug=True는 오류가 발생했을 때 디버깅 정보를 더 자세히 보여줍니다.
iface.launch(server_port=7861, server_name="0.0.0.0", debug=True)


In [ ]:
# Gradio 인터페이스를 종료합니다.
iface.close()


## 10.10 URL RAG 입력 예시

Gradio 앱에서 사용자는 다음 두 값을 입력합니다.

1. URL
2. 질문

예시:

```text
URL: https://example.com/some-article
질문: 이 페이지의 핵심 내용을 세 문장으로 요약해줘.
```

또는:

```text
URL: 제품 소개 페이지
질문: 이 제품의 주요 기능은 무엇인가요?
```

주의: 로그인이 필요한 페이지, JavaScript 렌더링 후 내용이 보이는 페이지, 크롤링 차단 페이지는 `WebBaseLoader`가 원하는 텍스트를 가져오지 못할 수 있습니다.


## 10.11 웹 문서 로딩의 한계

웹 URL 기반 RAG는 편리하지만 다음 한계가 있습니다.

### 1. JavaScript 렌더링 문제

많은 웹사이트는 브라우저에서 JavaScript를 실행한 뒤 내용을 표시합니다. 단순 HTML 로더는 이 내용을 가져오지 못할 수 있습니다.

### 2. 로그인 필요 페이지

로그인이 필요한 페이지는 기본 로더로 접근하기 어렵습니다.

### 3. 크롤링 차단

일부 사이트는 자동 접근을 차단할 수 있습니다.

### 4. 불필요한 텍스트 포함

헤더, 메뉴, 광고, 푸터 같은 불필요한 텍스트가 함께 들어올 수 있습니다.

### 5. 웹페이지 구조 변화

사이트 구조가 바뀌면 로딩 결과도 달라질 수 있습니다.

실무에서는 필요한 부분만 추출하도록 `bs_kwargs`, CSS selector, 전처리 로직을 더 정교하게 구성해야 합니다.


## 10.12 `bs_kwargs` 활용 방향

원본 노트북은 `bs_kwargs=dict()`처럼 빈 설정을 사용합니다.

```python
loader = WebBaseLoader(
    web_paths=(url,),
    bs_kwargs=dict()
)
```

필요하다면 BeautifulSoup 설정을 추가해 특정 태그나 클래스만 가져오도록 개선할 수 있습니다.

예를 들어 기사 본문이 특정 클래스에 들어 있다면 해당 부분만 추출하도록 조정할 수 있습니다.

이 교재에서는 원본 노트북의 흐름을 유지하되, 실무에서는 웹페이지 구조에 맞춘 전처리가 중요하다는 점을 기억하면 됩니다.


## 10.13 URL RAG 개선 포인트

URL RAG를 더 안정적으로 만들려면 다음을 개선할 수 있습니다.

### 1. URL 유효성 검사

사용자가 입력한 값이 실제 URL인지 먼저 확인합니다.

### 2. 예외 처리

웹페이지 로드 실패, 임베딩 실패, LLM 호출 실패를 각각 구분해 안내합니다.

### 3. 캐싱

같은 URL을 반복 질의할 때 매번 로드/임베딩하지 않도록 캐시를 둘 수 있습니다.

### 4. 출처 표시

답변과 함께 URL 또는 검색된 문서 일부를 표시하면 신뢰도가 올라갑니다.

### 5. context 길이 제한

검색 문서가 너무 많으면 LLM 입력이 길어집니다. Retriever의 `k` 값을 조정하거나 context를 요약할 수 있습니다.

### 6. 프롬프트 강화

`Context에 없는 내용은 답하지 마세요` 같은 규칙을 넣어 환각을 줄입니다.


## 10.14 자주 발생하는 오류와 해결 방법

### 1. URL 로드 실패

가능한 원인:

- URL 오타
- 인터넷 연결 문제
- 사이트 접근 차단
- 로그인 필요
- JavaScript 렌더링 필요

### 2. Ollama 임베딩 오류

확인할 것:

- Ollama 실행 여부
- `gemma2` 모델 설치 여부
- `ollama list` 결과

### 3. FAISS 생성 오류

확인할 것:

- `faiss-cpu` 설치 여부
- 로드된 문서가 비어 있지 않은지
- 임베딩 모델이 정상 응답하는지

### 4. Gradio 포트 충돌

해결:

- `iface.close()` 실행
- 커널 재시작
- `server_port=7862`처럼 다른 포트 사용

### 5. 답변 품질이 낮음

가능한 원인:

- 웹페이지에서 불필요한 텍스트가 많이 추출됨
- 검색된 청크가 질문과 관련 없음
- 프롬프트가 너무 약함
- 모델이 해당 언어나 도메인에 약함


## 10.15 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- URL 기반 RAG는 웹페이지를 외부 지식 소스로 사용하는 RAG입니다.
- `WebBaseLoader`로 웹페이지 내용을 LangChain Document로 읽을 수 있습니다.
- `RecursiveCharacterTextSplitter`로 웹 문서를 청크로 나눕니다.
- `OllamaEmbeddings`로 웹 문서 청크를 벡터화할 수 있습니다.
- FAISS 벡터스토어를 만들어 URL 문서 기반 검색을 수행합니다.
- 검색된 문서는 `format_docs()`로 하나의 context 문자열로 만듭니다.
- `prompt | llm | StrOutputParser()` 구조로 답변을 생성합니다.
- Gradio Blocks와 Tab을 사용해 Q/A 탭과 시각화 탭을 구성할 수 있습니다.
- 웹 RAG는 최신 정보 제공에 유용하지만, JavaScript 렌더링, 로그인, 크롤링 차단, 불필요한 텍스트 문제를 고려해야 합니다.

다음 장에서는 벡터스토어와 데이터베이스의 특징을 비교하고, 어떤 상황에서 어떤 저장소를 선택할지 정리합니다.
